# Figure 3: cascade contours in (z, alpha_th) for ER networks

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson, betabinom, beta as beta_dist
from figstyle import PALETTE, LW, REF_LW, FIGW_SINGLE, FIGW_DOUBLE, save

In [2]:
# Figure 3: kappa = 1 contours in (z, alpha_th), beta thresholds on ER graphs.
rho0 = 1e-9
alpha_vals = np.linspace(0.2, 10, 500)
z_vals = np.linspace(0.5, 25, 500)
B_th = 50
phi_show = [0.18, 0.25, 0.28, 0.3,0.8]
colors_show = PALETTE[:len(phi_show)]

kmax_az = int(np.floor(z_vals.max() + 10 * np.sqrt(z_vals.max()) + 10))
kk_az = np.arange(1, kmax_az + 1)
W = np.stack([kk_az * (kk_az - 1) / z * poisson.pmf(kk_az, z) for z in z_vals])

def kappa_grid(phi):
    m = alpha_vals / (alpha_vals + B_th)
    arg = np.minimum(1.0, m[:, None] / (kk_az[None, :] * phi))
    F = beta_dist.cdf(arg, alpha_vals[:, None], B_th)
    return (1 - rho0) * F @ W.T

fig, ax = plt.subplots(figsize=(FIGW_SINGLE, 2.7))
for phi, col in zip(phi_show, colors_show):
    K = kappa_grid(phi)
    ax.contour(z_vals, alpha_vals, K, levels=[1.0], colors=[col], linewidths=LW)
    ax.plot([], [], color=col, lw=LW, label=fr'$\bar\phi={phi}$')
ax.axhline(1.0, color="black", ls=":", lw=REF_LW, label=r'$\alpha_{\mathrm{th}}=1$')
ax.set_xlim(0, 15)
ax.set_ylim(0, 10)
ax.set_xlabel(r'mean degree $z=\langle k\rangle$')
ax.set_ylabel('threshold skew $\\alpha_{\\mathrm{th}}$')
ax.legend(loc="best", frameon=False, fontsize=7)
fig.tight_layout()
save(fig, "alphath_contour")
plt.show()

/var/folders/l3/dgm693wx7m3b1107vjtfb8k40000gq/T/ipykernel_34204/971567727.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# checks: kappa grows with z for alpha_th<1, decays for alpha_th>1, and the
# alpha_th=1 limit sits at beta_th/((1+beta_th)*phi)
a_lo = np.argmin(abs(alpha_vals - 0.5))
a_hi = np.argmin(abs(alpha_vals - 2.0))
a_one = np.argmin(abs(alpha_vals - 1.0))
z_lo = np.argmin(abs(z_vals - 5))
z_hi = np.argmin(abs(z_vals - 20))
K = kappa_grid(0.25)
print(f"alpha_th=0.5: kappa(z=5)={K[a_lo, z_lo]:.4f} -> kappa(z=20)={K[a_lo, z_hi]:.4f}")
print(f"alpha_th=2.0: kappa(z=5)={K[a_hi, z_lo]:.4f} -> kappa(z=20)={K[a_hi, z_hi]:.4f}")
assert K[a_lo, z_hi] > K[a_lo, z_lo], "alpha_th<1 should grow"
assert K[a_hi, z_hi] < K[a_hi, z_lo], "alpha_th>1 should decay"
pred = B_th / ((1 + B_th) * 0.25)
print(f"alpha_th=1.0: kappa(z=20)={K[a_one, z_hi]:.4f}, limit={pred:.4f}, ratio={K[a_one, z_hi] / pred:.4f}")
assert abs(K[a_one, z_hi] / pred - 1) < 0.2, "alpha_th=1 limit off"
print("figure 3 checks passed")

alpha_th=0.5: kappa(z=5)=2.8261 -> kappa(z=20)=6.7079
alpha_th=2.0: kappa(z=5)=1.7436 -> kappa(z=20)=1.0961
alpha_th=1.0: kappa(z=20)=3.3691, limit=3.9216, ratio=0.8591
figure 3 checks passed
